### Build End-to-End Pipeline Dataset: 
### Orders data with: 1)NULL values 2)String salary 3)Date column 
### Tasks: 1)Clean NULLs 2)Cast columns 3)Add derived columns 4)Apply UDF 
### Load: 1)Full load 2)Incremental load 3)Parameterize notebook

### 1. SAMPLE DATASET (Dirty + Realistic)
### 1)NULLs 2)String salary 3)Dates 4)Updates (for incremental load)
### from pyspark.sql import SparkSession

### spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()

### data = [
###     (1, "C001", "Laptop", "50000", "2024-01-01"),
###    (2, "C002", "Mobile", None, "2024-01-02"),
###    (3, "C003", "Tablet", "20000", "2024-01-03"),
###    (4, "C004", "Laptop", "55000", "2024-01-04"),
###    (5, "C005", "Headphones", None, "2024-01-05"),
###    ## Dont use this for first time (3, "C003", "Tablet", "22000", "2024-01-06"),  # updated
###    (6, "C006", "Camera", "30000", "2024-01-06"),
###    (7, "C007", "Mobile", "18000", "2024-01-07"),
###    (8, "C008", "Watch", "8000", "2024-01-07")
###]

### columns = ["order_id", "customer_id", "product", "amount", "updated_date"]

### df = spark.createDataFrame(data, columns)
### TASK LIST
### Task 1: Handle NULLs
### Replace NULL amount with 1000
### Task 2: Cast Columns
### Convert amount → integer
### Convert updated_date → date
### Task 3: Add Derived Columns
### bonus = amount * 0.1
### category:
### 20000 → High
### else → Low
### Task 4: Apply UDF
### Create amount_bucket:
### < 10000 → Low
### 10000–30000 → Medium
### 30000 → High
### Task 5: Full Load
### Load all data to target
### Task 6: Incremental Load
### Load only new/updated records
### Handle duplicates (keep latest)
### Task 7: Parameterization
### Pass:
### input path
last_loaded_date

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()
data = [
(1, "C001", "Laptop", "50000", "2024-01-01"),
(2, "C002", "Mobile", None, "2024-01-02"),
(3, "C003", "Tablet", "20000", "2024-01-03"),
(4, "C004", "Laptop", "55000", "2024-01-04"),
(5, "C005", "Headphones", None, "2024-01-05"),
(6, "C006", "Camera", "30000", "2024-01-06"),
(7, "C007", "Mobile", "18000", "2024-01-07"),
(8, "C008", "Watch", "8000", "2024-01-07")
]
columns = ["order_id", "customer_id", "product", "amount", "updated_date"]
df = spark.createDataFrame(data, columns)

**Task 1: Handle NULLs
Replace NULL amount with 1000**

In [0]:
df.fillna({'amount':'1000'}).display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
from pyspark.sql.functions import coalesce, lit

df.withColumn("amount", coalesce(df.amount, lit('1000'))).display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


**Task 2: Cast Columns
Convert amount → integer
Convert updated_date → date**

In [0]:
from pyspark.sql.functions import col, coalesce, lit

df = df.withColumn("amount", coalesce(col("amount").cast("int"), lit(1000))) \
       .withColumn("updated_date", col("updated_date").cast("date"))

df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


**Task 3: Add Derived Columns

bonus = amount * 0.1

category:
20000 → High
else → Low**

In [0]:
from pyspark.sql.functions import col, when

# Add both 'bonus' and 'category' columns
df = df.withColumn("bonus", col("amount") * 0.1) \
       .withColumn("category", when(col("amount") >= 20000, "High").otherwise("Low"))

# Use display() to see the new columns
df.display()

order_id,customer_id,product,amount,updated_date,bonus,category
1,C001,Laptop,50000,2024-01-01,5000.0,High
2,C002,Mobile,1000,2024-01-02,100.0,Low
3,C003,Tablet,20000,2024-01-03,2000.0,High
4,C004,Laptop,55000,2024-01-04,5500.0,High
5,C005,Headphones,1000,2024-01-05,100.0,Low
6,C006,Camera,30000,2024-01-06,3000.0,High
7,C007,Mobile,18000,2024-01-07,1800.0,Low
8,C008,Watch,8000,2024-01-07,800.0,Low


**Task 4: Apply UDF
Create amount_bucket:
< 10000 → Low
10000–30000 → Medium
30000 → High**

In [0]:
def amount_bucket(amount):
    if amount<10000:
        return "Low"
    elif amount>10000 and amount<30000:
        return "Medium"
    else:
        return "High"


In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

amount_bucket_udf = udf(amount_bucket, StringType())

In [0]:
df=df.withColumn("amount_bucket",amount_bucket_udf(col("amount")))
df.display()

order_id,customer_id,product,amount,updated_date,bonus,category,amount_bucket
1,C001,Laptop,50000,2024-01-01,5000.0,High,High
2,C002,Mobile,1000,2024-01-02,100.0,Low,Low
3,C003,Tablet,20000,2024-01-03,2000.0,High,Medium
4,C004,Laptop,55000,2024-01-04,5500.0,High,High
5,C005,Headphones,1000,2024-01-05,100.0,Low,Low
6,C006,Camera,30000,2024-01-06,3000.0,High,High
7,C007,Mobile,18000,2024-01-07,1800.0,Low,Medium
8,C008,Watch,8000,2024-01-07,800.0,Low,Low


**Task 5: Full Load
Load all data to target**

In [0]:
# 1. Define a table name (this will show up in your Catalog)
table_name = "orders_full_load"
df.write.format("delta").mode("overwrite").saveAsTable(table_name)
print(f"Full Load completed. Data saved to table: {table_name}")

# 3. Read it back to verify
spark.table(table_name).display()

Full Load completed. Data saved to table: orders_full_load


order_id,customer_id,product,amount,updated_date,bonus,category,amount_bucket
1,C001,Laptop,50000,2024-01-01,5000.0,High,High
2,C002,Mobile,1000,2024-01-02,100.0,Low,Low
3,C003,Tablet,20000,2024-01-03,2000.0,High,Medium
4,C004,Laptop,55000,2024-01-04,5500.0,High,High
5,C005,Headphones,1000,2024-01-05,100.0,Low,Low
6,C006,Camera,30000,2024-01-06,3000.0,High,High
7,C007,Mobile,18000,2024-01-07,1800.0,Low,Medium
8,C008,Watch,8000,2024-01-07,800.0,Low,Low


**Task 6: Incremental Load
Load only new/updated records
Handle duplicates (keep latest)**

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

# 1. Prepare the new data
new_data = [(3, "C003", "Tablet", "22000", "2024-01-06")]
incremental_df = spark.createDataFrame(new_data, columns)

# 2. Apply transformations so schema matches the target table
incremental_df = incremental_df.withColumn("amount", col("amount").cast("int")) \
                               .withColumn("updated_date", col("updated_date").cast("date")) \
                               .withColumn("bonus", col("amount") * 0.1) \
                               .withColumn("category", when(col("amount") >= 20000, "High").otherwise("Low")) \
                               .withColumn("amount_bucket", amount_bucket_udf(col("amount")))

# 3. Use Delta Merge to Upsert (Update if exists, Insert if new)
target_delta_table = DeltaTable.forName(spark, "orders_full_load")

target_delta_table.alias("target") \
  .merge(incremental_df.alias("source"), "target.order_id = source.order_id") \
  .whenMatchedUpdateAll() \
  .whenNotMatchedInsertAll() \
  .execute()

# 4. Check the results
spark.table("orders_full_load").orderBy("order_id").display()

order_id,customer_id,product,amount,updated_date,bonus,category,amount_bucket
1,C001,Laptop,50000,2024-01-01,5000.0,High,High
2,C002,Mobile,1000,2024-01-02,100.0,Low,Low
3,C003,Tablet,22000,2024-01-06,2200.0,High,Medium
4,C004,Laptop,55000,2024-01-04,5500.0,High,High
5,C005,Headphones,1000,2024-01-05,100.0,Low,Low
6,C006,Camera,30000,2024-01-06,3000.0,High,High
7,C007,Mobile,18000,2024-01-07,1800.0,Low,Medium
8,C008,Watch,8000,2024-01-07,800.0,Low,Low


Task 7: Parameterization
Pass:
input path
last_loaded_date

In [0]:
# Create widgets for input path and last loaded date
dbutils.widgets.text("input_path", "/FileStore/tables/orders_data/", "Input Path")
dbutils.widgets.text("last_loaded_date", "2024-01-01", "Last Loaded Date")

In [0]:
# Get values from widgets
path = dbutils.widgets.get("input_path")
filter_date =     dbutils.widgets.get("last_loaded_date")

print(f"Processing data from: {path}")
print(f"Filtering for records newer than: {filter_date}")

# Example usage in a filter
# df_filtered = df.filter(col("updated_date") > filter_date)

Processing data from: /FileStore/tables/orders_data/
Filtering for records newer than: 2024-01-01
